In [1]:
# src, configs
!cp -r /kaggle/input/czii-src /kaggle/working; mv /kaggle/working/czii-src /kaggle/working/src
!ln -s /kaggle/input/czii-configs /kaggle/working/configs

# train logs
!mkdir -p /kaggle/working/logs/train/runs
!ln -s /kaggle/input/all-data-250101-hard-r05-ftp-pre-1221-env2b2/250101-particle_hard_masks_r0.5-focalTverskyPp-pretrained_241221_299-hengck23_tf_efficientnetv2_b2_d64_256-s64_128-lr1e-3_decay05-bs4_2_2-ep100-transV3-preV4 /kaggle/working/logs/train/runs/
# !ln -s  /kaggle/working/logs/train/runs/

# for rootutils
!touch /kaggle/working/.project-root

In [2]:
%%capture
!cd /kaggle/input/czii-pip-packages-v2; pip install --no-index --find-links=./packages -r requirements.txt

In [3]:
experiment = "250101-particle_hard_masks_r0.5-focalTverskyPp-pretrained_241221_299-hengck23_tf_efficientnetv2_b2_d64_256-s64_128-lr1e-3_decay05-bs4_2_2-ep100-transV3-preV4"

batch_size_pred = 4
fp16_mode = True

ckpt_type = "last" # "last" or "best"

In [4]:
import hydra
import rootutils
import joblib
import os
from lightning import LightningDataModule, LightningModule
from hydra import compose, initialize
from pathlib import Path
from glob import glob
import numpy as np
import torch
import pandas as pd
from tqdm import tqdm
os.chdir("/kaggle/working/src")

rootutils.setup_root(Path().resolve(), indicator=".project-root", pythonpath=True)

root = os.environ["PROJECT_ROOT"]

In [5]:
import copick
from monai.data import MetaTensor

def get_ckpt_name(ckpt_type, fold):
    if ckpt_type == "last":
        if fold == 0:
            ckpt_name = "last.ckpt"
        else:
            ckpt_name = f"last-v{fold}.ckpt"
    elif ckpt_type == "best":
        ckpt_name = f"fold{fold}_epoch_*.ckpt"
    else:
        assert False, f"unknown ckpt_type: {ckpt_type}"    

    return ckpt_name


In [6]:
from copy import deepcopy
import torch_tensorrt
import torch
import gc

torch_tensorrt.runtime.set_multi_device_safe_mode(True)

print(f"experiment: {experiment}")
with initialize(version_base=None, config_path="configs"):
    cfg = compose(
        config_name="train",
        overrides=[f"experiment={experiment}"],
        return_hydra_config=True,
    )
    cfg.paths.output_dir = "${hydra.runtime.output_dir}"
    cfg.paths.work_dir =  "${hydra.runtime.cwd}"
    cfg.hydra.run.dir = cfg.log_dir
    cfg.hydra.runtime.output_dir = cfg.hydra.run.dir

if cfg.model.get("pretrained_ckpt_path"):
    cfg.model.pretrained_ckpt_path=None
if cfg.model.net.get("pretrained"):
    cfg.model.net.pretrained=False
if cfg.model.net.get("grad_checkpointing"):
    cfg.model.net.grad_checkpointing=False

ckpt_paths = []
ckpt_name = get_ckpt_name(ckpt_type, 0)
ckpt_paths.append(glob(os.path.join(cfg.paths.output_dir, "checkpoints", ckpt_name))[0])

# モデルの設定
model: LightningModule = hydra.utils.instantiate(cfg.model)
model.eval()
if fp16_mode:
    model.half()
    use_dtype = torch.float16
else:
    use_dtype = torch.float32

input_shape = [batch_size_pred, cfg.model.net.in_chans] + cfg.data.volume_size
inputs = [torch.randn(tuple(input_shape), dtype=use_dtype).cuda()]

for ckpt_path in ckpt_paths:
    model_fold = deepcopy(model)
    state_dict = torch.load(ckpt_path)["state_dict"]
    if cfg.model.compile:
        state_dict = {k.replace("_orig_mod.", ""): v for k, v in state_dict.items()}
    model_fold.load_state_dict(state_dict)
    model_fold.to("cuda")
    
    # trt_ep is a torch.fx.GraphModule object
    print(f"compiling all_data")
    trt_gm = torch_tensorrt.compile(
        model_fold, 
        ir="dynamo", 
        inputs=inputs, 
        enabled_precisions=[use_dtype],
        # torch_executed_ops={"torch.ops.aten.convolution.default"}
    )
    save_path = f"/kaggle/working/trt_{experiment}_all_data_{ckpt_type}_bs{batch_size_pred}_fp16{fp16_mode}.ep"
    torch_tensorrt.save(trt_gm, save_path, inputs=inputs)
    print(f"saved {save_path}")

    # GPUメモリの解放
    del model_fold  # model_foldオブジェクトの削除
    del trt_gm       # trt_gmオブジェクトの削除
    gc.collect()     # ガベージコレクションの実行
    torch.cuda.empty_cache()  # PyTorchのキャッシュをクリア
    print(f"Released GPU memory")


experiment: 250101-particle_hard_masks_r0.5-focalTverskyPp-pretrained_241221_299-hengck23_tf_efficientnetv2_b2_d64_256-s64_128-lr1e-3_decay05-bs4_2_2-ep100-transV3-preV4
compiling all_data
saved /kaggle/working/trt_250101-particle_hard_masks_r0.5-focalTverskyPp-pretrained_241221_299-hengck23_tf_efficientnetv2_b2_d64_256-s64_128-lr1e-3_decay05-bs4_2_2-ep100-transV3-preV4_all_data_last_bs4_fp16True.ep
Released GPU memory
